# Keystroke Dynamics ML — 4 Pipelines + Soft Voting

**Google Colab experiment:** Read your own `keystroke_features (1).csv`, compare **Random Forest, Logistic Regression, SVM and XGBoost**, and test a **Soft Voting** ensemble.

**Sequence:** Upload CSV → view features → split training/test → build leakage-safe Pipelines → tune with training-only 5-fold CV → select winner by CV → evaluate held-out data → export results.

**Note:** Keystroke timing is behavioral biometric data. Use only data you are authorized to process; do not post the raw CSV publicly. Accuracy on this small dataset is near chance and is not adequate for authentication.

## Step 1 — Install libraries

Colab usually includes most packages. This cell ensures **XGBoost** is available. Run this cell once.

In [ ]:
%pip -q install "xgboost>=2.0" "scikit-learn>=1.4" pandas numpy matplotlib joblib

## Step 2 — Upload and read the dataset

Run the cell below. If the CSV is already in the current folder, it loads automatically; otherwise, Colab opens the file picker. Upload **`keystroke_features (1).csv`**. This is the dataset-reading starting point.

In [ ]:
from pathlib import Path
import pandas as pd

known_names = ["keystroke_features (1).csv", "keystroke_features.csv"]
found = [Path(name) for name in known_names if Path(name).is_file()]
if found:
    csv_path = str(found[0])
else:
    from google.colab import files  # Runs in Google Colab
    uploaded = files.upload()
    csv_path = next((name for name in uploaded if name.lower().endswith('.csv')), None)
    if not csv_path:
        raise ValueError("Upload a keystroke CSV file to continue.")

df = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Shape:", df.shape)
display(df.head())

## Step 3 — Understand the features and target

The input columns are numerical timing statistics. The output `user_id` contains the person being classified. We must **never use `user_id` as a feature**.

In [ ]:
print("Columns:", df.columns.tolist())
print("Missing values:")
print(df.isna().sum())
print("User distribution:")
print(df["user_id"].value_counts())
display(df.describe(include='all').T)

## Step 4 — Import models and encode user IDs

Converting string user labels (`user_001`, etc.) to integers is necessary for multi-class XGBoost. Scaling is fitted **inside Pipelines**, not before splitting, so it cannot use held-out information.

In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
SEED = 42
TARGET = "user_id"

def read_keystroke_csv(path: str | Path):
    """Read the CSV and isolate numeric timing features and user labels."""
    df = pd.read_csv(path)
    if TARGET not in df.columns:
        raise ValueError(f"Expected '{TARGET}' column; found {df.columns.tolist()}")
    df = df.dropna(subset=[TARGET]).copy()
    features = df.drop(columns=[TARGET]).select_dtypes(include='number')
    features = features.replace([np.inf, -np.inf], np.nan)
    if features.empty or features.isna().all().any():
        raise ValueError('Need at least one numeric feature; no feature can be entirely missing.')
    encoder = LabelEncoder()
    labels = encoder.fit_transform(df[TARGET].astype(str))
    if len(encoder.classes_) < 2:
        raise ValueError('Need two or more users to train a classifier.')
    print(f'Dataset: {len(df)} samples | {features.shape[1]} numeric features | '
          f'{len(encoder.classes_)} users')
    print('Users:', dict(df[TARGET].value_counts()))
    print('Feature names:', features.columns.tolist())
    return features, labels, encoder

X, y, label_encoder = read_keystroke_csv(csv_path)
print("User ↔ integer encoding:", dict(enumerate(label_encoder.classes_)))

## Step 5 — Split the data (75% train / 25% test)

The **125 test samples** are kept out of cross-validation and parameter tuning. Stratification preserves approximately the same proportion of each user in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
print("Training rows:", X_train.shape[0])
print("Held-out test rows:", X_test.shape[0])
print("Total features:", X_train.shape[1])

## Step 6 — Create FOUR independent Scikit-learn Pipelines

| Pipeline | Preprocessing | Model |
|---|---|---|
| Random Forest | Median imputation | Random Forest |
| Logistic Regression | Median imputation + StandardScaler | Logistic Regression |
| SVM | Median imputation + StandardScaler | RBF SVM (probability enabled) |
| XGBoost | Median imputation | Multi-class gradient-boosted trees |

These are real `sklearn.pipeline.Pipeline` objects; all imputation and scaling occur **inside each cross-validation fold**.

In [ ]:
def make_models(n_classes: int):
    """All learned preprocessing is inside a Pipeline to avoid leakage."""
    return {
        'Random Forest': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestClassifier(random_state=SEED, n_jobs=1)),
        ]),
        'Logistic Regression': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=2000, random_state=SEED)),
        ]),
        'SVM': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', SVC(kernel='rbf', probability=True, random_state=SEED)),
        ]),
        'XGBoost': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', XGBClassifier(
                objective='multi:softprob' if n_classes > 2 else 'binary:logistic',
                eval_metric='mlogloss' if n_classes > 2 else 'logloss',
                tree_method='hist', random_state=SEED, n_jobs=1,
            )),
        ]),
    }


SEARCH_SPACES = {
    'Random Forest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 4, 8],
        'model__min_samples_leaf': [1, 3, 5],
    },
    'Logistic Regression': {
        'model__C': [0.01, 0.1, 1.0, 10.0],
        'model__class_weight': [None, 'balanced'],
    },
    'SVM': {
        'model__C': [0.1, 1.0, 10.0],
        'model__gamma': ['scale', 0.01, 0.1],
        'model__class_weight': [None, 'balanced'],
    },
    'XGBoost': {
        'model__n_estimators': [100, 180, 260],
        'model__max_depth': [2, 3, 5],
        'model__learning_rate': [0.03, 0.1],
        'model__subsample': [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0],
    },
}

models = make_models(n_classes=len(label_encoder.classes_))
for name, model in models.items():
    print(f"{name}: {list(model.named_steps)}")

## Step 7 — Tune and compare models using five-fold cross-validation

`RandomizedSearchCV` explores **4 hyperparameter configurations per algorithm** (increase to 10 or more for a longer search). We use only **training folds** for selection. The four tuned Pipelines are then combined using `VotingClassifier(voting="soft")`. The final choice is based on training CV—not on test accuracy.

Training can take longer with XGBoost and four-model voting.

In [ ]:
def train_and_compare(X_train, y_train, *, n_iter=4):
    """Tune the four Pipelines using training-set CV only; compare voting."""
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    best_pipelines = {}
    cv_rows = []
    for name, pipe in make_models(n_classes=len(np.unique(y_train))).items():
        print(f'\nTuning {name} with 5-fold CV ...', flush=True)
        search = RandomizedSearchCV(
            pipe, SEARCH_SPACES[name], n_iter=n_iter,
            scoring='accuracy', cv=folds, n_jobs=1,
            random_state=SEED, refit=True, error_score='raise',
        )
        search.fit(X_train, y_train)
        best_pipelines[name] = search.best_estimator_
        cv_rows.append({
            'model': name, 'cv_accuracy': float(search.best_score_),
            'parameters': str(search.best_params_),
        })
        print(f"{name}: CV accuracy = {search.best_score_:.1%}")
        print('  parameters:', search.best_params_)

    voters = [('rf', clone(best_pipelines['Random Forest'])),
              ('lr', clone(best_pipelines['Logistic Regression'])),
              ('svm', clone(best_pipelines['SVM'])),
              ('xgb', clone(best_pipelines['XGBoost']))]
    ensemble = VotingClassifier(estimators=voters, voting='soft', n_jobs=1)
    print('\nEvaluating four-model Soft Voting ensemble with 5-fold CV...', flush=True)
    vote_scores = cross_val_score(ensemble, X_train, y_train, cv=folds, scoring='accuracy', n_jobs=1)
    cv_rows.append({'model': 'Soft Voting (all four)',
                    'cv_accuracy': float(vote_scores.mean()),
                    'parameters': 'Four tuned Pipelines; equal soft-vote weights'})
    print('Soft Voting: CV accuracy =', f'{vote_scores.mean():.1%}')
    cv_table = pd.DataFrame(cv_rows).sort_values('cv_accuracy', ascending=False).reset_index(drop=True)
    print('\nCV COMPARISON (training data only):')
    print(cv_table[['model', 'cv_accuracy']].to_string(index=False))
    winner = cv_table.iloc[0]['model']
    print(f'\nModel chosen by training CV: {winner}')
    candidates = {**best_pipelines, 'Soft Voting (all four)': ensemble}
    return candidates, cv_table, winner

candidates, cv_table, selected_name = train_and_compare(
    X_train, y_train, n_iter=4
)
display(cv_table[["model", "cv_accuracy", "parameters"]])

## Step 8 — Plot CV accuracy (selection data)

The dotted reference is **20% random-chance accuracy** for five evenly likely users. The actual majority-class baseline is calculated from held-out data in the next step.

In [ ]:
ax = cv_table.sort_values("cv_accuracy").plot.barh(
    x='model', y='cv_accuracy', legend=False, figsize=(9, 4.5)
)
ax.axvline(1/len(label_encoder.classes_), linestyle=':', linewidth=1,
           label='Uniform random chance (20%)')
ax.set_xlabel('Mean 5-fold training CV accuracy')
ax.set_xlim(0, max(.35, cv_table.cv_accuracy.max()+.05))
ax.set_title('Keystroke classification: four Pipelines + Voting')
ax.legend()
plt.tight_layout()
plt.show()

## Step 9 — Evaluate the untouched hold-out and save results

All five candidates are reported for transparency, but the saved final Pipeline is the **one chosen in Step 7 by training CV**. Metrics include accuracy, balanced accuracy, macro-F1, confusion matrix, and per-user precision/recall.

**Security caveat:** A low-performing classifier should not be deployed for user verification.

In [ ]:
def evaluate(X_train, X_test, y_train, y_test, encoder, candidates, cv_table,
             winner, output_dir='outputs', feature_names=None):
    """Evaluate each candidate only after selection; save the CV-selected model."""
    out = Path(output_dir)
    out.mkdir(exist_ok=True, parents=True)
    baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
    baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
    print(f'\nMajority-class holdout baseline: {baseline_acc:.1%}')

    test_rows = []
    selected = None
    selected_pred = None
    for name, candidate in candidates.items():
        fitted = clone(candidate).fit(X_train, y_train)
        predictions = fitted.predict(X_test)
        test_rows.append({
            'model': name, 'test_accuracy': accuracy_score(y_test, predictions),
            'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
            'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
        })
        if name == winner:
            selected = fitted
            selected_pred = predictions

    test_table = pd.DataFrame(test_rows).sort_values('test_accuracy', ascending=False)
    print('\nHOLDOUT PERFORMANCE (for reporting, NOT model selection):')
    print(test_table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    selected_acc = accuracy_score(y_test, selected_pred)
    print(f'\nCV-selected model: {winner} | held-out test accuracy: {selected_acc:.1%}')
    print('\nPer-user report:')
    print(classification_report(
        y_test, selected_pred, labels=np.arange(len(encoder.classes_)),
        target_names=encoder.classes_, zero_division=0,
    ))
    ConfusionMatrixDisplay.from_predictions(
        encoder.inverse_transform(y_test), encoder.inverse_transform(selected_pred),
        labels=encoder.classes_, xticks_rotation=45, cmap='Blues', colorbar=False,
    )
    plt.title(f'CV-selected model: {winner}')
    plt.tight_layout()
    plt.savefig(out / 'keystroke_confusion_matrix.png', dpi=160)
    plt.close()

    cv_table.to_csv(out / 'keystroke_cv_scores.csv', index=False)
    test_table.to_csv(out / 'keystroke_holdout_scores.csv', index=False)
    model_file = out / 'keystroke_best_pipeline.joblib'
    joblib.dump({'pipeline': selected, 'label_encoder': encoder,
                 'features': list(feature_names if feature_names is not None else X_train.columns)}, model_file)
    result = {'selected_by_training_cv': winner, 'heldout_test_accuracy': float(selected_acc),
              'majority_holdout_baseline': float(baseline_acc),
              'training_rows': len(X_train), 'test_rows': len(X_test),
              'user_classes': encoder.classes_.tolist()}
    (out / 'keystroke_metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print('\nSample held-out prediction:',
          encoder.inverse_transform(selected.predict(X_test.iloc[[0]]))[0])
    print(f'Saved metrics, confusion matrix and CV-selected Pipeline to {out}/')
    print('Important: this small dataset may NOT support reliable user identification.')
    return result

metrics = evaluate(
    X_train, X_test, y_train, y_test, label_encoder,
    candidates, cv_table, selected_name, output_dir="outputs",
    feature_names=X.columns,
)
print(metrics)

## Step 10 — Predict one example from the held-out data

Reload only the Pipeline you just trained yourself. **Never load `.joblib` files from untrusted sources**, because joblib/pickle can execute code.

In [ ]:
artifact = joblib.load('outputs/keystroke_best_pipeline.joblib')
sample = X_test.iloc[[0]][artifact['features']]
prediction_num = artifact['pipeline'].predict(sample)
prediction_user = artifact['label_encoder'].inverse_transform(prediction_num)[0]
actual_user = label_encoder.inverse_transform(y_test[:1])[0]
print('Actual user   :', actual_user)
print('Predicted user:', prediction_user)

## Step 11 — Download experiment outputs

The `outputs/` folder includes CSV tables for CV and test metrics, the selected model, a JSON summary, and the confusion matrix image. Keep the trained model and the timing dataset private.

In [ ]:
import shutil
archive_path = shutil.make_archive('keystroke_four_model_results', 'zip', 'outputs')
print('Created:', archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print('Running locally; see the ZIP in the current folder.')

## Step 12 — Save the notebook to GitHub

The notebook and standalone script belong in the existing **private** repository [Ruchika-choudhary8/new](https://github.com/Ruchika-choudhary8/new). You can use **File → Save a copy in GitHub** inside Colab after signing in, or work from a local clone:

```bash
git clone https://github.com/Ruchika-choudhary8/new.git
cd new
# Save updated notebook and Python code into this folder
git add Keystroke_RF_LR_SVM_XGBoost_Colab.ipynb keystroke_four_models.py README.md requirements.txt .gitignore
git commit -m "Add keystroke RF LR SVM and XGBoost Pipelines"
git push origin main
```

Do **not** commit the keystroke CSV, user-identifying timing samples, secret tokens, or saved biometric classifiers. The repository's `.gitignore` excludes those files.

### Reproducible interpretation

With 500 samples and eight aggregate timing features, 5-way user identification performs near the chance/baseline level. **Higher model complexity alone does not make these features more discriminative.** For an authentication-ready study, collect multiple independent typing sessions, more key-pair-level signals, and evaluate using session-separated splits with FAR/FRR/EER.